Layer: Transformation → Silver
Source: Bronze (Delta Tables)
Target: Cleaned Silver Table


 1 Read Bronze Table


In [0]:
df_bronze_inventory = spark.read.table("workspace.bronze.erp_inventory") 

In [0]:
# Preview data
display(df_bronze_inventory)

# Check schema
df_bronze_inventory.printSchema()

#count check
df_bronze_inventory.count()

In [0]:
df_bronze_inventory.columns

 NOTE:
 Column naming follows best practices:
- snake_case format
- lowercase letters only
- descriptive and explicit names
   e.g., product_id, warehouse_id,reorder_level


 2 Data Type Casting


In [0]:
import pyspark.sql.functions as F

df_typed = df_bronze_inventory \
    .withColumn("product_id", F.col("product_id").cast("string")) \
    .withColumn("warehouse_id", F.col("warehouse_id").cast("string")) \
    .withColumn("stock_quantity", F.col("stock_quantity").cast("double")) \
    .withColumn("reorder_level", F.col("reorder_level").cast("int")) \
    .withColumn("last_updated", F.to_timestamp(F.col("last_updated"))) \
    .withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("timestamp")) \
    .withColumn("source_file", F.col("source_file").cast("string"))

df_typed.printSchema()


 3 Handle Null Values



 DATA QUALITY CONSTRAINTS (ERP - Invoices)

#
 Business rules applied in Silver layer:
#
Constraints:
- product_id cannot be NULL
- warehouse_id cannot be NULL
- stock_quantity >= 0
- reorder_level >= 0
- last_updated must be a valid date


In [0]:
df_clean = df_typed \
    .filter(F.col("product_id").isNotNull()) \
    .filter(F.col("warehouse_id").isNotNull()) \
    .filter(F.col("last_updated").isNotNull()) \
    .withColumn(
        "stock_quantity",
        F.when(F.col("stock_quantity").isNull(), 0).otherwise(F.col("stock_quantity"))
    ) \
    .withColumn(
        "reorder_level",
        F.when(F.col("reorder_level").isNull(), 0).otherwise(F.col("reorder_level"))
    )


 4 Data Quality Rules Enforcement


In [0]:
df_valid = df_clean \
    .filter(F.col("stock_quantity") >= 0) \
    .filter(F.col("reorder_level") >= 0)


 5 Deduplication 


In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("product_id", "warehouse_id") \
                    .orderBy(F.col("last_updated").desc())

df_dedup = df_valid \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

In [0]:
df_dedup.count()
df_dedup.display()


 6 Write to Silver Table


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS silver.inventory (
    product_id STRING,
    warehouse_id STRING,
    stock_quantity DOUBLE,
    reorder_level INT,
    last_updated TIMESTAMP,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
USING DELTA
""")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "silver.inventory")

(
    delta_table.alias("target")
    .merge(
        df_dedup.alias("source"),
        """
        target.product_id = source.product_id
        AND target.warehouse_id = source.warehouse_id
        """
    )
    .whenMatchedUpdate(
        condition = "source.last_updated > target.last_updated",
        set = {
            "stock_quantity": "source.stock_quantity",
            "reorder_level": "source.reorder_level",
            "last_updated": "source.last_updated",
            "ingestion_timestamp": "source.ingestion_timestamp",
            "source_file": "source.source_file"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.read.table("silver.inventory").count()
display(table("silver.inventory"))